### Baby script for fixing labels on mismatched trials

In [76]:
import pandas as pd
import numpy as np 
from matplotlib import pyplot as plt
import matplotlib

In [77]:
# load
cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Is_Correct"] 
true_labels = pd.read_excel("final_scores_16042026_v4.xlsx")

# files to relabel 
features_all = pd.read_pickle("training_features_18042026.pkl")
features_all = features_all.rename(columns={'Nap Number': 'Nap'})

trial_info = pd.read_csv("Trial_information_narcolepsy.csv", usecols=cols_inc) 

In [78]:
true_labels.head()

,Subject,Nap,Triggers_Order_Nap,Epoch,Muscle_type,Response_start_sample,Response_end_sample,Contraction_number
0,NL03JV,1,30,1.0,Corr,157.0,1943.0,0.0
1,NL03JV,1,30,1.0,Zygo,222.0,736.0,3.0
2,NL03JV,1,40,2.0,Corr,12.0,2233.0,0.0
3,NL03JV,1,40,2.0,Zygo,1449.0,1739.0,1.0
4,NL03JV,1,50,3.0,Corr,47.0,2188.0,0.0


Relabeling the files

In [79]:
key_cols = ['Subject', 'Nap', 'Triggers_Order_Nap']

# making true labels df match the organization of the mismatch column df 
true_labels_wide = (
    true_labels[true_labels['Muscle_type'].isin(['Zygo', 'Corr'])]
    .pivot_table(
        index=key_cols,
        columns='Muscle_type',
        values='Contraction_number',
        aggfunc='first'   
    )
    .reset_index()
    .rename(columns={
        'Zygo': 'Num_Contractions_Zygo',
        'Corr': 'Num_Contractions_Corr'
    })
)


In [80]:
df_all_temp = df_all[df_all['Num_Contractions_Corr_y'].notna()]
df_all_temp[df_all_temp['Subject'] == 'NL03JV']
#df_all_temp.head(50)

KeyError: 'Num_Contractions_Corr_y'

In [82]:
true_labels_wide.head()
df_all = features_all.merge(true_labels_wide, on=key_cols, how='left')

# checking mismatches 
zygo_mismatch = (
    df_all['Num_Contractions_Zygo_y'].notna() &
    (df_all['Num_Contractions_Zygo_y'] != df_all['Num_Contractions_Zygo_x'])
)

corr_mismatch = (
    df_all['Num_Contractions_Corr_y'].notna() &
    (df_all['Num_Contractions_Corr_y'] != df_all['Num_Contractions_Corr_x'])
)

df_all.loc[zygo_mismatch, 'Num_Contractions_Zygo_x'] = df_all.loc[zygo_mismatch, 'Num_Contractions_Zygo_y'].astype(int)
df_all.loc[corr_mismatch, 'Num_Contractions_Corr_x'] = df_all.loc[corr_mismatch, 'Num_Contractions_Corr_y'].astype(int)
 
changed_rows = df_all[zygo_mismatch | corr_mismatch].copy()
print("Number of rows updated:", len(changed_rows))
print(changed_rows[
    key_cols +
    ['Num_Contractions_Zygo_x', 'Num_Contractions_Corr_x',
     'Num_Contractions_Zygo_y', 'Num_Contractions_Zygo_y']
].head())

# fix df organization 
df_all = df_all.rename(columns={
    'Num_Contractions_Zygo_x': 'Num_Contractions_Zygo',
    'Num_Contractions_Corr_x': 'Num_Contractions_Corr',
    'Nap': 'Nap Number'
}).drop(columns=[
    'Num_Contractions_Zygo_y',
    'Num_Contractions_Corr_y'
])



Number of rows updated: 117
    Subject Nap Triggers_Order_Nap Num_Contractions_Zygo_x  \
13   RL19RS   1                 14                       0   
15   RL19RS   1                 16                       4   
83   NL04NF   3                 24                       0   
230  RL19RS   2                 51                       1   
324  RL23ET   4                 25                       0   

    Num_Contractions_Corr_x  Num_Contractions_Zygo_y  Num_Contractions_Zygo_y  
13                        2                      0.0                      0.0  
15                        1                      4.0                      4.0  
83                        0                      0.0                      0.0  
230                       0                      1.0                      1.0  
324                       1                      0.0                      0.0  


In [83]:
df_all.to_pickle("training_features_18042026.pkl")
df_all.to_excel("training_features_18042026.xlsx", index=False)